In [1]:
from pathlib import Path
import pandas as pd
import logging
import json

from gsm_benchmarker.results_analyser.prompt_result import PromptResult
from gsm_benchmarker.results_analyser.plotting_utils import Colour
from gsm_benchmarker.results_analyser.utils import pandas_to_latex


logger = logging.getLogger('notebook')


In [2]:
result_kwargs = dict(metric='correct')
pp = Path("../../../data/gsm-symbolic/outputs").resolve()


gsm_result = PromptResult(
    pp / "noq_default_full__12_05/final",
    colour=Colour('green'),
    full_label="GSM prompt",
    **result_kwargs
)

MODELS = gsm_result.mres.models


100%|██████████| 2/2 [00:00<00:00,  2.01it/s]


In [3]:
with open('mirzadeh-data.json') as f:
    original_results_df = pd.DataFrame(json.load(f))
    original_results_df = original_results_df.set_index('model')
    original_results_df = original_results_df[original_results_df.index.isin(MODELS)]

In [4]:
full_summary_df = gsm_result.mres.summary_data

In [5]:
PM = "±"

ours_gsm8k = full_summary_df[('GSM8K', 'accuracy')]
ours_main_mean = full_summary_df[('main', 'accuracy')]
ours_main_std = full_summary_df[('main', 'std')]
d = [f"{mean:.1f} ({PM} {std:.2f})"  for mean, std in zip(ours_main_mean, ours_main_std)]


df_combined = pd.DataFrame({
    "Mirzadeh - GSM8K": original_results_df['GSM8K_100'].apply(lambda v: f"{v:.1f}"),
    "Mirzadeh - main": original_results_df['symbolic_raw'],
    "Mirzadeh - diff": (original_results_df['GSM8K_100'] - original_results_df['symbolic_mean']).apply(lambda v: f"{v:.1f}"),
    "Ours - GSM8K": ours_gsm8k.apply(lambda v: f"{v:.1f}"),
    "Ours - main": pd.Series(d, index=ours_gsm8k.index),
    "Ours - diff": (ours_gsm8k - ours_main_mean).apply(lambda v: f"{v:.1f}")
})
df_combined.index.name = "Model"

model_order = original_results_df.index.tolist()

df_combined = df_combined.sort_index(
        key=lambda col: col.map({model: index for index, model in enumerate(model_order)})
)

In [6]:
(original_results_df['GSM8K_100'] - ours_gsm8k).abs().max()

22.0

In [7]:
(original_results_df['symbolic_mean'] - ours_main_mean).abs().max()


27.159999999999997

In [8]:
diff_diff = df_combined['Ours - diff'].astype(float) - df_combined['Mirzadeh - diff'].astype(float)
diff_diff

Model
gemma-2b                       4.2
gemma-2b-it                   -0.3
gemma-7b                     -24.1
gemma-7b-it                    0.8
gemma-2-2b                    -2.8
gemma-2-2b-it                 -5.7
gemma-2-9b                     0.0
gemma-2-9b-it                 -4.5
gemma-2-27b-it                -1.3
phi-2                          8.7
Phi-3-mini-128k-instruct      -3.0
Phi-3-medium-128k-instruct    -3.7
Phi-3.5-mini-instruct          1.7
Mistral-7B-v0.1               -1.9
Mistral-7B-Instruct-v0.1      -3.0
Mistral-7B-v0.3               -6.5
Mistral-7B-Instruct-v0.3       0.7
Mathstral-7B-v0.1              0.9
Meta-Llama-3-8B               21.2
Meta-Llama-3-8B-Instruct       7.7
dtype: float64

In [9]:
diff_diff.mean(), diff_diff.median(), diff_diff.std(), diff_diff.abs().max()

(np.float64(-0.5449999999999999),
 np.float64(-0.8),
 np.float64(8.34035686476178),
 np.float64(24.099999999999998))

In [10]:
print(pandas_to_latex(df_combined[[c for c in df_combined.columns if 'diff' in c]]))

\begin{table}[t]
\begin{tabular}{lcc}
\toprule
 & Mirzadeh - diff & Ours - diff \\
\midrule
gemma-2b & 2.8 & 7.0 \\
gemma-2b-it & 2.8 & 2.5 \\
gemma-7b & 24.4 & 0.3 \\
gemma-7b-it & 7.4 & 8.2 \\
gemma-2-2b & 5.9 & 3.1 \\
gemma-2-2b-it & 5.9 & 0.2 \\
gemma-2-9b & 7.9 & 7.9 \\
gemma-2-9b-it & 7.9 & 3.4 \\
gemma-2-27b-it & 3.7 & 2.4 \\
phi-2 & 11.6 & 20.3 \\
Phi-3-mini-128k-instruct & 4.3 & 1.3 \\
Phi-3-medium-128k-instruct & 6.5 & 2.8 \\
Phi-3.5-mini-instruct & 5.9 & 7.6 \\
Mistral-7B-v0.1 & 6.9 & 5.0 \\
Mistral-7B-Instruct-v0.1 & 11.5 & 8.5 \\
Mistral-7B-v0.3 & 4.0 & -2.5 \\
Mistral-7B-Instruct-v0.3 & 6.0 & 6.7 \\
Mathstral-7B-v0.1 & 6.0 & 6.9 \\
Meta-Llama-3-8B & -13.6 & 7.6 \\
Meta-Llama-3-8B-Instruct & -0.6 & 7.1 \\
\bottomrule
\end{tabular}
\end{table}

